In [ ]:
import logging
from pathlib import Path

import myo
import myoktros
import numpy as np
import pandas as pd
import tensorflow as tf
from matplotlib import pyplot as plt
from myo.types import EMGMode
from myoktros import Gesture
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split

In [ ]:
# global variables
BATCH_SIZE = 100
EPOCHS = 1000
N_SENSORS = 8
arm_dominance = "right"
assets = Path('.') / "assets"
data_path = Path('.') / "data"
emg_mode = myo.types.EMGMode.SEND_FILT
n_samples = 50
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)
np.set_printoptions(precision=3, suppress=True)

In [ ]:
ksm = myoktros.KerasSequentialModel(
    arm_dominance,
    assets,
    emg_mode,
    n_samples,
)

In [ ]:
test_data_path = Path('.') / "tests" / "data"

x_test = myoktros.GestureModel.read_data_agg(test_data_path, arm_dominance, emg_mode, n_samples)
y_test = x_test.pop('gesture')

print(x_test.shape, y_test.shape)

predictions = ksm.model.predict(x_test)
predicted_labels = np.argmax(predictions, axis=1)

print(predicted_labels.shape)

# normalize="pred": 
cm = confusion_matrix(y_test, predicted_labels, normalize="pred")

# labels are gestures
legend = [g.name for g in myoktros.Gesture]

plt.imshow(cm)
plt.ylabel("Actual")
plt.xlabel("Predicted")
plt.yticks(np.arange(len(legend)), legend)
plt.xticks(np.arange(len(legend)), legend, rotation='vertical')
plt.colorbar()